# GRASP coupling data
Coupling simulations in the TICRA\'s *General Reflector Antenna Software Package* (GRASP) typically comprise several transmitters and receivers. 
In addition to a fixed configuration, the entity of receivers may be translated/rotated relatively to the transmitters. These translations/rotations are specified by the *Movement Definition*.

The output of the simulation is stored in the following formats:
- GRASP coupling data in cuts: `.cut` (implemented for file format "Cuts" and $\leq2$ movement definitions, not implemented for single cut file format "One Cut")
- GRASP coupling data in grids: `.grd` (not implemented)

The `pycra-tools` package reads data into [Xarray](https://docs.xarray.dev/en/stable/) labeled multi-dimensional array. 
To correctly interpret the numerical data, additional information must be included. 
For instance, the files lack any frequency information. Moreover, only the coordinates of the last two movement definitions are stored in the file (in the case of the "Cuts" file format).

The missing information can be retrieved from the `.tor` file associated with the project, which is the only method currently implemented.

## GRASP coupling data `.cut` files with file format "Cuts"
The coupling quotients are written as separate cuts in the file (blocks separated by the label "coupling" and one line of information).
The header (only) contains information about the two movement definitions that are last specified in *Movement Definition*.

Let $R_i(x)$ denote the operator that describes the $i$--th translation/rotation, where the variable $x$ describes parameterizes the movement (distance along / angle around specific axis). Denote $R_{ij}=R_i(x_j)$ the operation for a specific value $x=x_j$ . The `.cut` files look about as follows (with notation `{"fast coordinate" vs. coupling quotients}` representing a cut):

In [ ]:
Example: Zero movement definitions.
- nr. cuts = 1
- "slow coordinates": none
- "fast coordinate": none
- nr. of lines per cut = len(freqs)
- Missing information: freqs
-------------------------------------------------
| {freq vs. coupling quotients}                 |
-------------------------------------------------

Example: One movement definition (translation/rotation)
- nr. cuts = len(freqs)
- "slow coordinates": none
- "fast coordinate": R1 (values indicated by V_INI, V_INC, V_NUM)
- nr. of lines per cut = len((R11, R12, ...))
- Missing information: freqs
-------------------------------------------------
| freq1 {R1 vs. coupling quotients}             |
| freq2 {R1 vs. coupling quotients}             |
| ...                                           |
-------------------------------------------------

Example: Two movement definitions (e.g. translation and rotation)
- nr. cuts = len(freq) * len((R11, R12, ...))
- "slow coordinate": R1 (value indicated with dummy variable C)
- "fast coordinate": R2 (values indicated by V_INI, V_INC, V_NUM)
- nr. of lines per cut = len((R21, R22, ...))
- Missing information: freqs
- Analogy to field data in cuts:
  freq -> freq
  fix (e.g. phi) -> slow
  varying (e.g. theta) -> fast
-------------------------------------------------
| freq1 - R11 {R2 vs. coupling quotients}       |
|       - R12 {R2 vs. coupling quotients}       |
|       - ...                                   |
| freq2 - R11 {R2 vs. coupling quotients}       |
|       - R12 {R2 vs. coupling quotients}       |
|       - ...                                   |
| ...                                           |
-------------------------------------------------

Example: Three movement definitions
- nr. cuts = len(freq) * len((R11, R12, ...)) * len((R21, R22, ...))
- "slow coordinates": R1 (value not indicated), R2 (value indicated with dummy variable C)
- "fast coordinate": R3 (values indicated by V_INI, V_INC, V_NUM)
- nr. of lines per cut = len((R31, R32, ...))
- Missing information: freqs, (R11, R12, ...)
-------------------------------------------------
| freq1 - R11 - R21 {R3 vs. coupling quotients} |
|             - R22 {R3 vs. coupling quotients} |
|             - ...                             |
|       - R12 - R21 {R3 vs. coupling quotients} |
|             - R22 {R3 vs. coupling quotients} |
|             - ...                             |
|       - ...                                   |
| freq2 - R11 - R21 {R3 vs. coupling quotients} |
|             - R22 {R3 vs. coupling quotients} |
|             - ...                             |
|       - R12 - R21 {R3 vs. coupling quotients} |
|             - R22 {R3 vs. coupling quotients} |
|             - ...                             |
|       - ...                                   |
| ...                                           |
-------------------------------------------------


The imported data will look as follows:

In [ ]:
<xarray.DataArray '/home/phjschmid/phdphysics/code/projects/packaging_tutorial/pycra-tools/example_data/coupling/coupling_and_plate/Job_01/coupling_system_horn2.cut' (
freq: 3,                                                                                                                                                        m1: 2)> Size: 480B
array([[[-6.28695461e-02+1.45969926e-01j,
          2.93096156e-17+4.46127276e-17j],
        [-3.70121490e-02+3.74765850e-02j,
         -2.86842634e-17+7.85025440e-18j],
        [-1.57488024e-03-3.30759392e-02j,
         -7.11963966e-18-6.18384019e-18j],
        [ 5.41676391e-03-1.24236078e-02j,
          6.00732943e-18-5.04613994e-18j],
        [ 3.44965208e-03-2.36274447e-03j,
         -7.27946118e-18-2.90158518e-18j]],

       [[ 1.64107078e-01+1.22012966e-01j,
          3.09979223e-17-7.55052393e-17j],
        [-4.03973535e-02+2.51242766e-02j,
          1.21418069e-17-9.50897270e-19j],
        [ 6.85938012e-03-9.88623749e-03j,
         -1.30547149e-17+8.82181045e-19j],
        [ 1.53183653e-02+4.32051038e-03j,
         -5.02357712e-18-2.24942624e-18j],
        [ 3.25302819e-03+3.20507690e-03j,
          8.71610665e-18-6.74883187e-18j]],

       [[ 1.73424698e-01-1.39732156e-01j,
         -2.13587114e-16+1.81427661e-16j],
        [-1.35313533e-02+6.33697935e-02j,
          5.29452067e-17-9.62717736e-17j],
        [ 8.07594618e-03-1.87586532e-02j,
          9.70886841e-18+1.87788403e-17j],
        [-3.38991027e-03+1.51830847e-02j,
          1.37207916e-18-9.73826973e-18j],
        [-2.98901523e-03+3.62899183e-03j,
          1.62641114e-17-4.50354255e-18j]]])
Coordinates:
  * freq     (freq) float64 24B 3.5e+09 5.25e+09 7e+09
  * m1       (m1) float64 16B 0.0 90.0
Attributes:
    file_name:            /home/phjschmid/phdphysics/code/projects/packaging_...
    class_name:           coupling_system
    file_form:            cuts
    receiver_sources:     ['rectangular_horn2']
    amplitude_only:       off
    movement_definition:  movement_definition_horn2
    cqlist:               off
    comment:              None